# 02 — Supervised Fine-Tuning (SFT) Baseline

**RLHF Pipeline for Compact Open-Source LLMs**

This notebook fine-tunes a compact base model on the chosen responses from HH-RLHF using LoRA adapters. The resulting SFT model serves as the baseline for RLHF comparison (RQ1) and as the starting policy for PPO training.

## Design Choices

- **Base model:** Qwen2.5-0.5B — compact enough for single-GPU or CPU training
- **Fine-tuning method:** LoRA (Low-Rank Adaptation) via PEFT — trains <1% of parameters
- **SFT data:** Chosen responses from HH-RLHF, used as a lightweight instruction-following proxy
- **Framework:** TRL SFTTrainer for stable, well-tested training

## 2.1 Environment and Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.data_utils import set_seed, load_dataset_from_disk
from src.model_utils import load_causal_lm, load_tokenizer, apply_lora, build_lora_config, save_model_and_tokenizer
from src.sft_train import save_training_log

set_seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration
MODEL_NAME     = "Qwen/Qwen2.5-0.5B"
MAX_SEQ_LENGTH = 512
OUTPUT_DIR     = PROJECT_ROOT / "outputs" / "models" / "sft_qwen"
LOG_DIR        = PROJECT_ROOT / "outputs" / "logs"
FIGURES_DIR    = PROJECT_ROOT / "outputs" / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Minimal config dict for model loading
cfg = {
    "seed": 42,
    "training": {"fp16": True, "bf16": False, "gradient_checkpointing": True},
    "lora": {
        "r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
        "target_modules": ["q_proj", "v_proj"], "bias": "none",
    },
}

## 2.2 Load Processed SFT Data

In [ ]:
sft_train = load_dataset_from_disk(PROJECT_ROOT / "data" / "processed" / "sft" / "train")
sft_val   = load_dataset_from_disk(PROJECT_ROOT / "data" / "processed" / "sft" / "validation")

print(f"SFT train: {len(sft_train)} samples, columns: {sft_train.column_names}")
print(f"SFT val:   {len(sft_val)} samples")
print(f"\nSample text (first 300 chars):\n{sft_train[0]['text'][:300]}")

## 2.3 Model and Tokenizer Setup

In [ ]:
tokenizer = load_tokenizer(MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH)
model     = load_causal_lm(MODEL_NAME, cfg)

## 2.4 LoRA Configuration

We use LoRA to fine-tune only a small subset of parameters:
- **r=16:** Rank of the low-rank matrices
- **lora_alpha=32:** Scaling factor (alpha/r = 2)
- **target_modules=["q_proj", "v_proj"]:** Attention query and value projections
- **lora_dropout=0.05:** Light regularisation

In [ ]:
lora_config = build_lora_config(cfg, task_type="CAUSAL_LM")
model = apply_lora(model, lora_config)

# PEFT + gradient checkpointing compatibility
use_grad_ckpt = torch.cuda.is_available()
if use_grad_ckpt:
    model.enable_input_require_grads()
    print("Gradient checkpointing enabled with input_require_grads")
else:
    print("Gradient checkpointing disabled (CPU)")

model.train()

## 2.5 SFT Training

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

use_fp16 = cfg["training"]["fp16"] and torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    evaluation_strategy="steps",
    save_total_limit=2,
    fp16=use_fp16,
    gradient_checkpointing=use_grad_ckpt,
    report_to="none",
    seed=42,
    remove_unused_columns=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_train,
    eval_dataset=sft_val,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

print("SFTTrainer created. Starting training...")

In [ ]:
# Run this cell to start actual training
trainer.train()

# Save model and log
save_model_and_tokenizer(model, tokenizer, OUTPUT_DIR)
save_training_log(trainer.state.log_history, LOG_DIR / "sft_training_log.csv")
print(f"SFT model saved to {OUTPUT_DIR}")

## 2.6 Training Loss Curve

Load the training log and plot the loss curve.

In [ ]:
log_path = LOG_DIR / "sft_training_log.csv"

if log_path.exists():
    log_df = pd.read_csv(log_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Training loss
    train_rows = log_df.dropna(subset=["loss"])
    if not train_rows.empty:
        axes[0].plot(train_rows["step"], train_rows["loss"])
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("SFT Training Loss")
    
    # Eval loss
    eval_rows = log_df.dropna(subset=["eval_loss"])
    if not eval_rows.empty:
        axes[1].plot(eval_rows["step"], eval_rows["eval_loss"], marker="o", markersize=4)
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Eval Loss")
        axes[1].set_title("SFT Evaluation Loss")
    
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "sft_loss_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("TO BE FILLED AFTER RUNNING FULL SFT TRAINING")

## 2.7 Baseline Generation Examples

Generate a few sample outputs from the SFT model.

In [ ]:
from src.inference import generate_responses

eval_prompts = [
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
    "How can I improve my sleep quality?",
]

if OUTPUT_DIR.exists() and (OUTPUT_DIR / "adapter_config.json").exists():
    model.eval()
    responses = generate_responses(
        model, tokenizer, eval_prompts,
        max_new_tokens=128,
        prompt_template="### Human:\n{prompt}\n\n### Assistant:\n",
    )
    for prompt, response in zip(eval_prompts, responses):
        print(f"PROMPT: {prompt}")
        print(f"RESPONSE: {response[:300]}")
        print("-" * 60)
else:
    print("TO BE FILLED AFTER RUNNING SFT TRAINING")

## Summary

| Output | Location |
|---|---|
| SFT LoRA adapter | `outputs/models/sft_qwen/` |
| Training log CSV | `outputs/logs/sft_training_log.csv` |
| Loss curve plot | `outputs/figures/sft_loss_curves.png` |

**Next:** Proceed to `03_reward_model.ipynb` for reward model training.